# Entrenamiento v2 — TF-IDF + Features manuales

En este notebook mejoramos el modelo baseline añadiendo features manuales al TF-IDF.

**Baseline (notebook 3):** LogisticRegression + TF-IDF → F1-macro validación: 0.87

**Mejoras en este notebook:**
1. Features manuales: longitud de texto + conteo de keywords de dominio por clase
2. Concatenación TF-IDF + features manuales
3. Re-entrenamiento de LogisticRegression
4. Comparativa con baseline
5. Registro en MLflow (Experimento 1)

In [11]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [12]:
import sys
import os

# Localizar src/classifier/ de forma robusta y ajustar cwd al directorio
# de este notebook para que rutas relativas (datasets/, data/, model/) funcionen
# independientemente de desde donde se lance Jupyter/VS Code.
_cwd = os.getcwd()
_candidates = [
    os.path.join(_cwd, "src", "classifier"),
    os.path.abspath(".."),
    os.path.abspath("."),
]
for _p in _candidates:
    if os.path.isfile(os.path.join(_p, "functions.py")):
        if _p not in sys.path:
            sys.path.insert(0, _p)
        # Cambiar cwd al directorio de este notebook
        os.chdir(os.path.join(_p, "classifier_dataset_artificial"))
        break

import functions  # noqa: E402
functions.MLFLOW_EXPERIMENT = "clasificador_riesgo_ia_artificial"
functions._DATASET_TAGS = {"dataset_type": "artificial", "dataset_source": "eu_ai_act_flagged"}

## 1. Carga de datos

In [13]:
import pandas as pd

train_df = pd.read_csv("data/processed/train.csv")
val_df = pd.read_csv("data/processed/validation.csv")
test_df = pd.read_csv("data/processed/test.csv")

X_train = train_df["text_final"]
y_train = train_df["etiqueta"]
X_val = val_df["text_final"]
y_val = val_df["etiqueta"]
X_test = test_df["text_final"]
y_test = test_df["etiqueta"]

print(f"Train: {len(X_train)} muestras")
print(f"Validation: {len(X_val)} muestras")
print(f"Test: {len(X_test)} muestras")

Train: 210 muestras
Validation: 45 muestras
Test: 45 muestras


## 2. Vectorización TF-IDF (misma config que baseline)

In [14]:
from functions import crear_tfidf

tfidf, X_train_tfidf, X_val_tfidf, X_test_tfidf = crear_tfidf(
    X_train, X_val, X_test,
    max_features=5000,
    ngram_range=(1, 2),
)

Vocabulario TF-IDF: 3773 términos
Forma train: (210, 3773)
Forma validation: (45, 3773)
Forma test: (45, 3773)


## 3. Features manuales

In [15]:
from functions import crear_features_manuales, combinar_features, KEYWORDS_DOMINIO

# Generar features manuales para cada conjunto
feat_train = crear_features_manuales(X_train)
feat_val = crear_features_manuales(X_val)
feat_test = crear_features_manuales(X_test)

print(f"Features manuales: {list(feat_train.columns)}")
print(f"Forma: {feat_train.shape}")
print()
print(feat_train.describe().round(2))

Features manuales: ['num_palabras', 'num_caracteres', 'kw_inaceptable', 'kw_alto_riesgo', 'kw_riesgo_limitado', 'kw_riesgo_minimo', 'kw_salvaguarda']
Forma: (210, 7)

       num_palabras  num_caracteres  kw_inaceptable  kw_alto_riesgo  \
count        210.00          210.00          210.00          210.00   
mean          15.10          138.98            0.20            0.23   
std            1.63           15.20            0.48            0.49   
min           11.00           96.00            0.00            0.00   
25%           14.00          129.25            0.00            0.00   
50%           15.00          140.00            0.00            0.00   
75%           16.00          149.00            0.00            0.00   
max           19.00          183.00            2.00            3.00   

       kw_riesgo_limitado  kw_riesgo_minimo  kw_salvaguarda  
count              210.00            210.00          210.00  
mean                 0.32              0.18            0.48  
std    

In [16]:
# Concatenar TF-IDF + features manuales
X_train_combined = combinar_features(X_train_tfidf, feat_train)
X_val_combined = combinar_features(X_val_tfidf, feat_val)
X_test_combined = combinar_features(X_test_tfidf, feat_test)

print(f"Forma train combinada: {X_train_combined.shape}")
print(f"Forma validation combinada: {X_val_combined.shape}")
print(f"Forma test combinada: {X_test_combined.shape}")
print(f"  (TF-IDF: {X_train_tfidf.shape[1]} + manuales: {feat_train.shape[1]} = {X_train_combined.shape[1]})")

Forma train combinada: (210, 3780)
Forma validation combinada: (45, 3780)
Forma test combinada: (45, 3780)
  (TF-IDF: 3773 + manuales: 7 = 3780)


## 4. Entrenamiento LogisticRegression con features combinadas

In [17]:
from functions import entrenar_modelo_baseline

modelo_v2 = entrenar_modelo_baseline(X_train_combined, y_train, X_val_combined, y_val)

=== Resultados en VALIDACIÓN ===

                 precision    recall  f1-score   support

    alto_riesgo       0.72      0.93      0.81        14
    inaceptable       0.89      0.73      0.80        11
riesgo_limitado       1.00      0.90      0.95        10
  riesgo_minimo       0.89      0.80      0.84        10

       accuracy                           0.84        45
      macro avg       0.88      0.84      0.85        45
   weighted avg       0.86      0.84      0.85        45

F1-score macro (validación): 0.8505


## 5. Comparativa con baseline

In [18]:
from sklearn.metrics import f1_score

# Métricas del baseline (notebook 3)
BASELINE_F1_MACRO = 0.8698
BASELINE_ACCURACY = 0.87

# Métricas del modelo v2
y_val_pred_v2 = modelo_v2.predict(X_val_combined)
f1_v2 = f1_score(y_val, y_val_pred_v2, average="macro")
acc_v2 = (y_val_pred_v2 == y_val).mean()

print("=== COMPARATIVA VALIDACIÓN ===")
print(f"{'Métrica':<20} {'Baseline':>10} {'V2 (+ features)':>16} {'Diferencia':>12}")
print("-" * 60)
print(f"{'F1-macro':<20} {BASELINE_F1_MACRO:>10.4f} {f1_v2:>16.4f} {f1_v2 - BASELINE_F1_MACRO:>+12.4f}")
print(f"{'Accuracy':<20} {BASELINE_ACCURACY:>10.4f} {acc_v2:>16.4f} {acc_v2 - BASELINE_ACCURACY:>+12.4f}")

=== COMPARATIVA VALIDACIÓN ===
Métrica                Baseline  V2 (+ features)   Diferencia
------------------------------------------------------------
F1-macro                 0.8698           0.8505      -0.0193
Accuracy                 0.8700           0.8444      -0.0256


## 6. Guardar artefactos

In [19]:
import joblib
import os

output_dir = "model"
os.makedirs(output_dir, exist_ok=True)

joblib.dump(modelo_v2, os.path.join(output_dir, "modelo_logreg_v2.joblib"))
joblib.dump(modelo_v2, os.path.join(output_dir, "modelo_clasificador.joblib"))
joblib.dump(tfidf, os.path.join(output_dir, "tfidf_vectorizer.joblib"))

print("Modelo v2 guardado en: model/modelo_logreg_v2.joblib")
print("Modelo v2 guardado en: model/modelo_clasificador.joblib")
print("Vectorizador guardado en: model/tfidf_vectorizer.joblib")

Modelo v2 guardado en: model/modelo_logreg_v2.joblib
Modelo v2 guardado en: model/modelo_clasificador.joblib
Vectorizador guardado en: model/tfidf_vectorizer.joblib


## 7. Registro en MLflow — Experimento 1

In [20]:
# ── MLflow (solo falla esta celda si el servidor no está disponible) ──
import mlflow
from functions import configure_mlflow, MLFLOW_EXPERIMENT

try:
    configure_mlflow()
    mlflow.set_experiment(MLFLOW_EXPERIMENT)

    with mlflow.start_run(run_name="v2_tfidf_features_manuales"):
        mlflow.log_param("tfidf_max_features",     5000)
        mlflow.log_param("tfidf_ngram_range",      "(1, 2)")
        mlflow.log_param("tfidf_sublinear_tf",     True)
        mlflow.log_param("modelo",                 "LogisticRegression")
        mlflow.log_param("max_iter",               1000)
        mlflow.log_param("features_manuales",      list(feat_train.columns))
        mlflow.log_param("n_keywords_por_clase",   len(list(KEYWORDS_DOMINIO.values())[0]))
        mlflow.log_param("total_features",         X_train_combined.shape[1])

        mlflow.log_metric("val_f1_macro",          f1_v2)
        mlflow.log_metric("val_accuracy",          acc_v2)
        mlflow.log_metric("baseline_f1_macro",     BASELINE_F1_MACRO)

        print("✓ Exp 1 registrado en MLflow")
        print(f"  Val F1-macro: {f1_v2:.4f}")
        print(f"  Run ID: {mlflow.active_run().info.run_id}")
except Exception as e:
    print(f"⚠ MLflow no disponible: {e}")

Password obtenida desde variable de entorno local.
MLflow configurado correctamente → https://18.201.64.41/


c:\Users\rammu\anaconda3\envs\venv_proyecto\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host '18.201.64.41'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
c:\Users\rammu\anaconda3\envs\venv_proyecto\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host '18.201.64.41'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
c:\Users\rammu\anaconda3\envs\venv_proyecto\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host '18.201.64.41'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
c:\Users\r

✓ Exp 1 registrado en MLflow
  Val F1-macro: 0.8505
  Run ID: 0c05a2b15f9a476caee5b0ec8b44f8a7


c:\Users\rammu\anaconda3\envs\venv_proyecto\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host '18.201.64.41'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
